## Import libraries

In [1]:
import numpy as np
import pandas as pd
from obspy import read, UTCDateTime
from glob import glob
from tqdm import tqdm

## Load Hypoinverse location result

In [2]:
hypinv_catalog=pd.read_csv("/home/msseo/works/10.Earthquake_cycle_project/Gwangyang_sequence/1.HypoInv/kim1983/Gwangyang.sum", sep=',')
hypinv_catalog

,DATE TIME,SEC,LAT,LON,DEPTH,PREF,MAG,NM,NUM,GAP,...,QASR,ID-NUM,LOC,DUR,MAG1,AMP,MAG1.1,EXT,MAG,Unnamed: 23
0,2021/07/20 16:14,18.84,34.9985,127.7273,14.43,,,,65,36,...,B,200000,,,,,,,,NaN
1,2021/08/27 00:23,15.47,35.0006,127.7278,14.69,,,,96,35,...,B,200001,,,,,,,,NaN
2,2021/08/27 22:03,23.17,35.0000,127.7273,14.51,,,,104,35,...,B,200002,,,,,,,,NaN
3,2021/08/28 13:34,11.21,34.9996,127.7286,14.67,,,,101,35,...,B,200003,,,,,,,,NaN
4,2021/08/28 13:39,22.80,34.9996,127.7295,14.23,,,,63,36,...,B,200004,,,,,,,,NaN
5,2021/08/28 23:45,56.75,34.9998,127.7287,14.48,,,,96,35,...,B,200005,,,,,,,,NaN
6,2026/01/01 02:49,47.51,34.9997,127.7271,14.82,,,,49,40,...,B,200006,,,,,,,,NaN
7,2026/01/01 08:47,59.93,34.9981,127.7269,14.87,,,,56,36,...,B,200007,,,,,,,,NaN
8,2026/02/02 19:41,19.89,34.9999,127.7279,14.41,,,,79,35,...,B,200008,,,,,,,,NaN
9,2026/02/03 02:15,35.73,34.9980,127.7293,14.41,,,,41,40,...,B,200009,,,,,,,,NaN


## Modify origin times and adjust P and S picks

In [3]:
eventdir_list=sorted(glob("/home/msseo/works/10.Earthquake_cycle_project/Gwangyang_sequence/waveforms_100km/20*"))
for i in tqdm(range(len(eventdir_list))):
    eventdir=eventdir_list[i]
    datetime=hypinv_catalog["   DATE     TIME"][i]
    second=hypinv_catalog[" SEC "][i]
    year, month, day=int(datetime[:4]), int(datetime[5:7]), int(datetime[8:10])
    hour, minute, second, microsecond=int(datetime[11:13]), int(datetime[14:16]), int(second), int((second-int(second))*1e6)
    origin_time=UTCDateTime(year, month, day, hour, minute, second, microsecond)
    sacfile_list=glob(eventdir+"/*.sac")
    
    for sacfile in sacfile_list:
        tr=read(sacfile)[0]
        
        try:
            ptime=tr.stats.starttime-tr.stats.sac.b+tr.stats.sac.a
        except:
            ptime=-9999
            
        try:
            stime=tr.stats.starttime-tr.stats.sac.b+tr.stats.sac.t0
        except:
            stime=-9999

        # Change origin time
        tr.stats.sac.nzyear=origin_time.year
        tr.stats.sac.nzjday=origin_time.julday
        tr.stats.sac.nzhour=origin_time.hour
        tr.stats.sac.nzmin=origin_time.minute
        tr.stats.sac.nzsec=origin_time.second
        tr.stats.sac.nzmsec=int(origin_time.microsecond/1000)
        
        # Adjust P/S picks
        if ptime != -9999:
            tr.stats.sac.a=ptime-origin_time
        else:
            pass
        
        if stime != -9999:
            tr.stats.sac.t0=stime-origin_time
        else:
            pass
        
        tr.write(sacfile, format="SAC")
        # print(tr.stats.sac.nzsec, tr.stats.sac.nzmsec)
        # print("\n")

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 11/11 [00:05<00:00,  1.86it/s]
